# 01_train_study

- Ablation study for loss function selection
- Each run trains DeclipNet with the same architecture and a different loss function configuration
- All runs trained to convergence using early stopping with relaxed patience
  (fewer epochs without improvement than in final training)
- Tracks train loss and val SI-SDR per epoch — loss curve used to assess convergence
- Loss functions compared via paired comparisons on val SI-SDR at convergence
- All run results saved with run name, loss config, train loss history, and val SI-SDR history
- Winning loss function carried forward to 01_train_final for full training with stricter convergence

In [1]:
import sys
import os
import json
import time
import subprocess
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from pathlib import Path

sys.path.insert(0, "..")
from config import *
from util import (
    DeclipDataset,
    DeclipNet,
    l1_loss,
    weighted_l1_loss,
    multires_stft_loss,
    dwt_loss,
    si_sdr, 
    make_loss_fn,
    train_run
)

In [2]:
# Hyperparameters
BATCH_SIZE = 32
LR = 1e-3
PATIENCE = 10        # epochs without improvement before early stopping (relaxed for study)
MIN_DELTA = 1e-3     # minimum SI-SDR improvement to count as progress
VAL_EVERY = 1        # evaluate val SI-SDR every N epochs

# Model
H = 8
N_ATTN = 3
NUM_HEADS = 4
FFN_DIM = 256

# Device
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: mps


In [3]:
# Dataset and dataloaders
train_dataset = DeclipDataset(TRAIN_OUT / "train_blocks.pt", TRAIN_OUT / "train_manifest.json")
val_dataset = DeclipDataset(VAL_OUT / "val_blocks.pt", VAL_OUT / "val_manifest.json")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")

Train samples: 225045
Val samples:   28131


## Loss Function Ablation Study

### Stage 1a — Waveform loss: does amplitude weighting help? ✓
| Run | Loss | Purpose |
|-----|------|---------|
| l1 | L1 | Baseline |
| weighted_l1 (p=1) | Weighted L1 | Does amplitude weighting help? |

Result: weighted_l1 wins (21.25 dB vs 18.67 dB). Amplitude weighting helps significantly.

### Stage 1b — Weight power tuning: how aggressive should weighting be?
| Run | Loss | Purpose |
|-----|------|---------|
| weighted_l1 (p=1) | Weighted L1 | Already complete |
| weighted_l1_p2 (p=2) | Weighted L1 | Is more aggressive weighting better? |
| weighted_l1_p4 (p=4) | Weighted L1 | Is even more aggressive weighting better? |

Winner carries forward as the waveform component in all subsequent stages.

### Stage 1c — Spectral loss: does it help, and which is better?
| Run | Loss | Purpose |
|-----|------|---------|
| winner + stft | Winner + STFT | Does spectral loss help? |
| winner + dwt | Winner + DWT | Is DWT better than STFT? |

### Stage 1d — Spectral loss: how many wavelet levels? (default = 4)
#### Only if DWT being used
| Run | Loss | Purpose |
|-----|------|---------|
| winner + dwt | Winner + DWT | Does 5 level DWT outperform default of 4? |

### Stage 2 — Frequency weighting: does it help for the winning spectral term?
| Run | Loss | Purpose |
|-----|------|---------|
| winner + stft_weighted | Winner + weighted STFT | (only if STFT wins stage 1c) |
| winner + dwt_weighted | Winner + weighted DWT | (only if DWT wins stage 1c) |

### Interpretation
- Stage 1a winner = amplitude weighting helps
- Stage 1b winner = best weight power
- Stage 1c winner = best spectral term
- STage 1d winner = best decomp method (for DWT only)
- Stage 2 winner = final loss function for 01_train_final

In [4]:
# # Stage 1a — waveform loss comparison
# run_configs_1a = [
#     {"name": "l1"},
#     {"name": "weighted_l1"},
# ]

# caffeinate = subprocess.Popen(["caffeinate", "-i"])
# try:
#     for config in run_configs_1a:
#         train_run(
#             run_name=config["name"],
#             loss_fn=make_loss_fn(config["name"]),
#             train_loader=train_loader,
#             val_loader=val_loader,
#             device=DEVICE,
#             lr=LR, 
#             patience=PATIENCE,
#             min_delta=MIN_DELTA,
#             h=H,
#             n_attn=N_ATTN,
#             num_heads=NUM_HEADS,
#             ffn_dim=FFN_DIM
#         )
# finally:
#     caffeinate.terminate()
#     print("Caffeinate terminated.")

In [5]:
# # Stage 1b — weight_power tuning: p=1 already complete, run p=2 and p=4
# run_configs_1b = [
#     {"name": "weighted_l1_p2"},
#     {"name": "weighted_l1_p4"},
# ]

# caffeinate = subprocess.Popen(["caffeinate", "-i"])
# try:
#     for config in run_configs_1b:
#         train_run(
#             run_name=config["name"],
#             loss_fn=make_loss_fn(config["name"]),
#             train_loader=train_loader,
#             val_loader=val_loader,
#             device=DEVICE,
#             lr=LR, 
#             patience=PATIENCE,
#             min_delta=MIN_DELTA,
#             h=H,
#             n_attn=N_ATTN,
#             num_heads=NUM_HEADS,
#             ffn_dim=FFN_DIM
#         )
# finally:
#     caffeinate.terminate()
#     print("Caffeinate terminated.")

In [6]:
# # Check relative magnitudes of loss terms
# model_check = DeclipNet(H=H, N=N_ATTN, num_heads=NUM_HEADS, ffn_dim=FFN_DIM).to(DEVICE)
# model_check.eval()

# clipped_sample, clean_sample = next(iter(train_loader))
# clipped_sample, clean_sample = clipped_sample.to(DEVICE), clean_sample.to(DEVICE)

# with torch.no_grad():
#     output_sample = model_check(clipped_sample)

# l1_val = weighted_l1_loss(output_sample, clean_sample, clipped_sample, weight_power=1.0).item()
# stft_val = multires_stft_loss(output_sample, clean_sample, weight_power=0.0).item()
# dwt_val = dwt_loss(output_sample, clean_sample, weight_power=0.0).item()

# print(f"weighted_l1:    {l1_val:.6f}")
# print(f"multires_stft:  {stft_val:.6f}")
# print(f"dwt:            {dwt_val:.6f}")
# print(f"stft / l1 ratio: {stft_val / l1_val:.2f}x")
# print(f"dwt  / l1 ratio: {dwt_val  / l1_val:.2f}x")

# del model_check

In [7]:
# # Stage 1c i — spectral loss comparison: weighted_l1 + stft
# run_configs_1c = [
#     {"name": "weighted_l1_stft"},
# ]

# caffeinate = subprocess.Popen(["caffeinate", "-i"])
# try:
#     for config in run_configs_1c:
#         train_run(
#             run_name=config["name"],
#             loss_fn=make_loss_fn(config["name"]),
#             train_loader=train_loader,
#             val_loader=val_loader,
#             device=DEVICE,
#             lr=LR,
#             patience=PATIENCE,
#             min_delta=MIN_DELTA
#         )
# finally:
#     caffeinate.terminate()
#     print("Caffeinate terminated.")

In [8]:
# # Stage 1c ii — spectral loss comparison: weighted_l1 + dwt
# run_configs_1c = [
#     {"name": "weighted_l1_dwt"},
# ]

# caffeinate = subprocess.Popen(["caffeinate", "-i"])
# try:
#     for config in run_configs_1c:
#         train_run(
#             run_name=config["name"],
#             loss_fn=make_loss_fn(config["name"]),
#             train_loader=train_loader,
#             val_loader=val_loader,
#             device=DEVICE,
#             lr=LR,
#             patience=PATIENCE,
#             min_delta=MIN_DELTA
#         )
# finally:
#     caffeinate.terminate()
#     print("Caffeinate terminated.")

In [9]:
# # Stage 1d — spectral loss comparison: dwt w 5 decomp levels
# run_configs_1c = [
#     {"name": "weighted_l1_dwt_level5"},
# ]

# caffeinate = subprocess.Popen(["caffeinate", "-i"])
# try:
#     for config in run_configs_1c:
#         train_run(
#             run_name=config["name"],
#             loss_fn=make_loss_fn(config["name"]),
#             train_loader=train_loader,
#             val_loader=val_loader,
#             device=DEVICE,
#             lr=LR,
#             patience=PATIENCE,
#             min_delta=MIN_DELTA
#         )
# finally:
#     caffeinate.terminate()
#     print("Caffeinate terminated.")

[weighted_l1_dwt_level5] epoch 001 | loss: 0.260331 | val SI-SDR: 14.6441 dB | best: 14.6441 dB | no improve: 0/10
[weighted_l1_dwt_level5] epoch 002 | loss: 0.191221 | val SI-SDR: 15.7576 dB | best: 15.7576 dB | no improve: 0/10
[weighted_l1_dwt_level5] epoch 003 | loss: 0.176117 | val SI-SDR: 16.6325 dB | best: 16.6325 dB | no improve: 0/10
[weighted_l1_dwt_level5] epoch 004 | loss: 0.166415 | val SI-SDR: 17.4209 dB | best: 17.4209 dB | no improve: 0/10
[weighted_l1_dwt_level5] epoch 005 | loss: 0.159629 | val SI-SDR: 17.9199 dB | best: 17.9199 dB | no improve: 0/10
[weighted_l1_dwt_level5] epoch 006 | loss: 0.155466 | val SI-SDR: 18.0279 dB | best: 18.0279 dB | no improve: 0/10
[weighted_l1_dwt_level5] epoch 007 | loss: 0.152463 | val SI-SDR: 18.3528 dB | best: 18.3528 dB | no improve: 0/10
[weighted_l1_dwt_level5] epoch 008 | loss: 0.151061 | val SI-SDR: 18.2655 dB | best: 18.3528 dB | no improve: 1/10
[weighted_l1_dwt_level5] epoch 009 | loss: 0.147840 | val SI-SDR: 18.8361 dB | b

KeyboardInterrupt: 